In [4]:
import os
from pathlib import Path
import json
from jsonargparse import CLI
import boto3

import time
from copy import deepcopy
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextGenerationPipeline, AutoModelForSequenceClassification
from peft import PeftModel
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from accelerate import Accelerator
import multiprocessing as mp
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_batch_response(texts, base_model, model_path, temperature, max_tokens, top_p):

    model = LLM(model=base_model,
                enable_lora=True,
                tensor_parallel_size=8,
                dtype="bfloat16",
                max_lora_rank=64)

    lora_req = LoRARequest("lora1",1,model_path)
    sampling_params = SamplingParams(max_tokens=max_tokens, temperature=temperature, top_p=top_p)

    results = model.generate(texts, sampling_params, lora_request=lora_req)

    completions = [o.outputs[0].text for o in results]

    return completions


def main(from_json: str = None, to_json: str = None, prompt: str = None, base_model: str = 'llama-3.1-instruct',
         model_path: str = 'llama-3.1-instruct', temperature: float = 0, max_tokens: int = 512, top_p=0.9,
         n_samples: int = -1, input_field: str = 'input', existing_json: str = None):
    EXSTING = {}
    if existing_json is not None:
        with open(existing_json, 'r') as f:
            for l in f.readlines():
                d = json.loads(l)
                if d['resp'] != 'API Failed':
                    EXSTING[d['prompt']] = d
    
    
    path = Path(to_json)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

    with open(from_json, "r") as fr, open(to_json, 'w') as fw:

        lines = fr.readlines()
        total_lines = min(len(lines), n_samples) if n_samples > 0 else len(lines)
        start_time = time.time()
           
        texts = [prompt.format(json.loads(lines[i])[input_field]) for i in range(len(lines))]
        
        results = get_batch_response(
                            texts, base_model, model_path, temperature, max_tokens, top_p
                        )
        
        for result in results:
            fw.write(json.dumps(result) + '\n')


dataset = "redial"
model = "llama-3.2-instruct"
local_folder = "test_epoch10_seed2"
alg = 'sft'

main(from_json='testsets/'+dataset+'/test.jsonl',
    to_json=f'test_res/{alg}/{dataset}/{model}/{dataset}_test_parallel.jsonl',
    prompt="Pretend you are a movie recommender system. I will give you a conversation between a user and you (a recommender system). Based on the conversation, reply 20 recommendations in the format of '1. [Movie Name]\n 2. [Movie Name]\n 3. [Movie Name]\n'. Then terminate the conversation. Here is the conversation: {}",
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    model_path=f'../../outputs/sft/{dataset}/{model}/{local_folder}',
    temperature=0.1,
    max_tokens=512,
    n_samples=-1)

INFO 07-22 01:43:41 [config.py:823] This model supports multiple tasks: {'classify', 'reward', 'embed', 'score', 'generate'}. Defaulting to 'generate'.


INFO 07-22 01:43:41 [config.py:1946] Defaulting to use mp for distributed inference
INFO 07-22 01:43:41 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-22 01:43:41 [core.py:455] Waiting for init message from front-end.
INFO 07-22 01:43:41 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=8, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), o

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


WARNING 07-22 01:43:41 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f36b0ea7c10>
(VllmWorker rank=0 pid=192813) INFO 07-22 01:43:41 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_8cffa025'), local_subscribe_addr='ipc:///tmp/6038edd5-9160-4eb3-8f07-17c212bbb9ce', remote_subscribe_addr=None, remote_addr_ipv6=False)
WARNING 07-22 01:43:41 [utils.py:2737] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7f36b0ea5d20>
(VllmWorker rank=1 pid=192814) INFO 07-22 01:43:41 [shm_broadcast.py:289] vLLM message queue communication handle: Handle(local_reader_ranks=[0], buffer_handle=(1, 10485760, 10, 'psm_2ba3045d'), local_subscribe_addr='ipc:///tmp/f5f2fd75-9c00-47ea-a28f

(VllmWorker rank=0 pid=192813) Exception ignored in: <finalize object at 0x7f3979f32080; dead>
(VllmWorker rank=0 pid=192813) Traceback (most recent call last):
(VllmWorker rank=0 pid=192813)   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/weakref.py", line 591, in __call__
(VllmWorker rank=0 pid=192813)     return info.func(*info.args, **(info.kwargs or {}))
(VllmWorker rank=0 pid=192813)   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core_client.py", line 313, in __call__
(VllmWorker rank=0 pid=192813)     self.engine_manager.close()
(VllmWorker rank=0 pid=192813)   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/utils.py", line 273, in close
(VllmWorker rank=0 pid=192813)     self._finalizer()
(VllmWorker rank=0 pid=192813)   File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/weakref.py", line 591, in __call__
(VllmWorker rank=0 pid=192813)     return info.func(*info.

...

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorker rank=2 pid=192815) INFO 07-22 01:43:47 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=1 pid=192814) (VllmWorker rank=6 pid=192824) INFO 07-22 01:43:47 [weight_utils.py:345] No model.safetensors.index.json found in remote.
INFO 07-22 01:43:47 [cuda.py:252] Using Flash Attention backend on V1 engine.
(VllmWorker rank=7 pid=192825) INFO 07-22 01:43:47 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=5 pid=192823) INFO 07-22 01:43:47 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=4 pid=192822) INFO 07-22 01:43:47 [weight_utils.py:292] Using model weights format ['*.safetensors']
(VllmWorker rank=0 pid=192813) INFO 07-22 01:43:47 [default_loader.py:272] Loading weights took 0.12 seconds
(VllmWorker rank=0 pid=192813) INFO 07-22 01:43:47 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorker rank=3 pid=192817) INFO 07-22 01:43:47 [weight_utils.py:345] No model.safeten

Exception ignored in: <finalize object at 0x7f3979f32080; dead>
Traceback (most recent call last):
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/weakref.py", line 591, in __call__
    return info.func(*info.args, **(info.kwargs or {}))
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/engine/core_client.py", line 313, in __call__
    self.engine_manager.close()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/utils.py", line 273, in close
    self._finalizer()
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/weakref.py", line 591, in __call__
    return info.func(*info.args, **(info.kwargs or {}))
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/site-packages/vllm/v1/utils.py", line 627, in shutdown
    if proc.is_alive():
  File "/home/sagemaker-user/.conda/envs/collabllm/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._pa

Adding requests:   0%|          | 0/3552 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3552 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…